#  Stacking Exploration
**Base learners:** XGBoost + CatBoost  
**Meta-learner:** Logistic Regression  
> Extensible à 5 modèles — voir section *Add more base learners*

## 0. Imports et setting

In [1]:
import os
os.chdir("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.pipeline import Pipeline

from src.data_processing.data_load import load_data_processed_from_S3
from src.models.model_saving import load_model_from_s3
from Config import S3_BUCKET_FINAL_MODELS, MLFLOW_EXPERIMENT_NAME


#Setting so catboost load succeed 
from src.models.train_catboost import CatBoostAutoCat #import catboost class created
import __main__
__main__.CatBoostAutoCat = CatBoostAutoCat  # inject class catboost in __main__

## 1. Load base models from S3

In [17]:
xgb_pipeline  = load_model_from_s3(bucket=S3_BUCKET_FINAL_MODELS, model_name="xgboost")
cat_pipeline   = load_model_from_s3(bucket=S3_BUCKET_FINAL_MODELS, model_name="catboost")

print("XGBoost steps :", list(xgb_pipeline.named_steps.keys()))
print("CatBoost steps:", list(cat_pipeline.named_steps.keys()))

xgb_model= xgb_pipeline["model"]

XGBoost steps : ['model']
CatBoost steps: ['model']


## 2. Load data (processed)

In [37]:
X_val=load_data_processed_from_S3("X_test_enc_nonorm_duplicatemissing")
y_val=load_data_processed_from_S3("X_test_enc_nonorm_duplicatemissing").values

## 3. Baseline — individual model performance

In [26]:
def evaluate(name, model, X, y):
    proba = model.predict_proba(X)[:, 1]
    auc   = roc_auc_score(y, proba)
    ap    = average_precision_score(y, proba)
    print(f"{name:<20} ROC-AUC: {auc:.4f}  |  Avg Precision: {ap:.4f}")
    return proba

print("=== Validation set ===")
p_xgb = evaluate("XGBoost", xgb_model, X_val, y_val)
#p_cat = evaluate("CatBoost", cat_pipeline, X_val, y_val)

=== Validation set ===


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:client_ro_souscripteur: object, client_departement: object, client_produit: object, client_garantie_base: object, client_garantie_base_plus_options: object, client_millesime_tarifaire_produit: object, client_zone_tarifaire: object, client_type_zone_tarifaire: object, client_mode_paiement: object, client_frequence_paiement: object, courtier_reseau_courtage: object, courtier_segmentation_interne: object, courtier_type_commission: object, age_categories: category, courtier_anciennete_categories: category, client_nps_category_n: object

In [36]:
# Vérifier les colonnes object restantes
print(X_val.select_dtypes(include='object').columns.tolist())

['client_ro_souscripteur', 'client_departement', 'client_produit', 'client_garantie_base', 'client_garantie_base_plus_options', 'client_millesime_tarifaire_produit', 'client_zone_tarifaire', 'client_type_zone_tarifaire', 'client_mode_paiement', 'client_frequence_paiement', 'courtier_reseau_courtage', 'courtier_segmentation_interne', 'courtier_type_commission', 'client_nps_category_n']


In [34]:
xgb_model.predict_proba(X_val)

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:client_ro_souscripteur: object, client_departement: object, client_produit: object, client_garantie_base: object, client_garantie_base_plus_options: object, client_millesime_tarifaire_produit: object, client_zone_tarifaire: object, client_type_zone_tarifaire: object, client_mode_paiement: object, client_frequence_paiement: object, courtier_reseau_courtage: object, courtier_segmentation_interne: object, courtier_type_commission: object, age_categories: category, courtier_anciennete_categories: category, client_nps_category_n: object

In [ ]:
evaluate()

## 4. Build Stacking

In [ ]:
# ── Base learners ──────────────────────────────────────────────────────
# On utilise les pipelines complets (preprocessing inclus)
base_learners = [
    ('xgboost',  xgb_pipeline),
    ('catboost', cat_pipeline), #add models here
]

# ── Meta-learner ───────────────────────────────────────────────────────
meta_learner = LogisticRegression(
    C=0.1,              # régularisation — important pour éviter l'overfit
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs',
    random_state=42,
)

# ── Stack ──────────────────────────────────────────────────────────────
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

stacking = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=cv_strategy,
    stack_method='predict_proba',   # passe les probas au meta-learner
    passthrough=False,              # mettre True pour passer aussi X brut au meta-learner
    n_jobs=-1,
)

print("Fitting stacking...")
stacking.fit(X_train, y_train)
print("✅ Done")

## 5. Evaluate stacking vs individual models

In [ ]:
print("=== Validation set ===")
p_xgb   = evaluate("XGBoost",   xgb_pipeline, X_val, y_val)
p_cat   = evaluate("CatBoost",  cat_pipeline,  X_val, y_val)
p_stack = evaluate("Stacking",  stacking,      X_val, y_val)

In [ ]:
# ROC + PR curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, proba in [("XGBoost", p_xgb), ("CatBoost", p_cat), ("Stacking", p_stack)]:
    RocCurveDisplay.from_predictions(y_val, proba, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_val, proba, name=name, ax=axes[1])

axes[0].set_title("ROC Curve")
axes[1].set_title("Precision-Recall Curve")
plt.tight_layout()
plt.show()

## 6. Meta-learner coefficients (interprétabilité)

In [ ]:
meta = stacking.final_estimator_
model_names = [name for name, _ in base_learners]

coef_df = pd.DataFrame({
    'model': model_names,
    'coef' : meta.coef_[0]
}).sort_values('coef', ascending=False)

print(coef_df.to_string(index=False))

coef_df.plot(kind='bar', x='model', y='coef', legend=False,
             title='Meta-learner weights per base model')
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 7. Cross-validation score (robustesse)


In [ ]:
# ⚠️ Long à tourner — commenter si pas nécessaire
cv_scores = cross_val_score(
    stacking, X_train, y_train,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1
)
print(f"CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

## 8. Save stacking model


In [ ]:
from src.models.model_saving import save_model_to_s3

save_model_to_s3(stacking, model_name='stacking_xgb_cat', bucket=S3_BUCKET_FINAL_MODELS)
print('✅ Stacking model saved')